# Сравнение сборщиков с эталоном — мышь ERP003950

Шесть образцов, только тяжёлая цепь IgG (IGH); UMI в датасете нет, каждый образец собирается отдельно.

**Эталон** — наблюдаемые последовательности V…J шаблонов симуляции (`00_primary_truth`, интервал `vj_start…vj_end`) с сохранёнными SHM; CDR3 эталона — AIRR `junction` исходных ридов.

**Recall (по шаблонам).** Каждая эталонная V…J выравнивается minimap2 на контиги сборщика; берётся лучшее выравнивание.

| класс | условие |
|---|---|
| `exact_full` | вся V…J, 0 замен и инделей — **основная метрика** |
| `one_diff_full` | вся V…J, 1 отличие |
| `near_full` | ≥95% V…J, идентичность ≥98% |
| `partial` | есть выравнивание, но хуже |
| `missing` | выравнивания нет |

**Проверенные шаблоны.** Recall, классы восстановимости и CDR3 считаются по шаблонам `verified=True` (человек — V…J подтверждена консенсусом UMI, мышь — ≥2 рида после схлопывания ошибок). Непроверенные шаблоны есть в библиотеке, но их последовательность может нести ошибку секвенирования исходных данных. Контиг, лучше всего совпавший с непроверенным шаблоном, получает класс precision `unverified_match` и ошибкой не считается. Recall по непроверенным шаблонам показан отдельной строкой `UNVERIFIED` (часть из них — ошибки исходных данных).

В эталоне много клональных родственников, отличающихся 1–7 SHM, поэтому мягкие классы засчитывают шаблону контиг родственника. **Отрицательный контроль** — шаблоны класса восстановимости `no_reads` (их риды не попали в FASTQ, собрать их нельзя): доля «восстановленных» среди них показывает частоту ложных совпадений на каждом уровне строгости.

**CDR3 (по клонотипам = локус + junction).** Клонотип найден, если его junction целиком содержится в контиге, на который выровнялся любой его шаблон. Контроль — клонотипы, ни один шаблон которых не прочитан.

**Precision (по контигам).** Каждый контиг выравнивается на эталонные V…J: `exact_full` — совпадает с эталонной V…J целиком без отличий; `near_full` — ≥95% V…J при идентичности ≥98%; `correct_partial` — ≥100 нт при идентичности ≥98%; `incorrect` — ниже (ошибки, химеры); `unmapped`.


## Окружение и параметры

In [ ]:
import os, sys, shutil, subprocess, time, gzip, json
from pathlib import Path

BCR_ENV = Path(os.environ.get("BCR_ENV", "/data/user/epishkin/conda/envs/bcr_env"))
os.environ["PATH"] = str(BCR_ENV / "bin") + os.pathsep + os.environ["PATH"]

DATASET = "ERP003950"
BRANCH = os.environ.get("BCR_BRANCH", "insilicoseq_150bp_novaseq_post_annotation_filtered_random_cut_amp_ec1x3_in5ng_rb3x_ss250")
SAMPLES = os.environ["BCR_SAMPLES"].split(",") if os.environ.get("BCR_SAMPLES") else ["ERR346596", "ERR346597", "ERR346598", "ERR346599", "ERR346600", "ERR346601"]
THREADS = int(os.environ.get("BCR_THREADS", 20))
FORCE = False

def find_root():
    cwd = Path.cwd().resolve()
    for root in (Path(os.environ.get("BCR_VOLUME", "/data/user/epishkin")), cwd, *cwd.parents):
        if (root / "results" / DATASET / "simulated" / BRANCH).is_dir():
            return root
    raise FileNotFoundError(f"results/{DATASET}/simulated/{BRANCH} not found; set BCR_VOLUME")

ROOT = find_root()
DATASET_DIR = ROOT / "results" / DATASET
BRANCH_DIR = DATASET_DIR / "simulated" / BRANCH
FASTQ_DIR = BRANCH_DIR / "06_fastq_pe150"
ASM_DIR = BRANCH_DIR / "assemblers"

def reads(sample):
    r1, r2 = FASTQ_DIR / f"{sample}_R1.fastq.gz", FASTQ_DIR / f"{sample}_R2.fastq.gz"
    if not (r1.exists() and r2.exists()):
        raise FileNotFoundError(f"simulated FASTQ missing for {sample}: {r1}")
    return r1, r2

def run_logged(cmd, log_path, cwd=None, heartbeat=300):
    log_path = Path(log_path); log_path.parent.mkdir(parents=True, exist_ok=True)
    print("RUN:", " ".join(map(str, cmd)), "\nLOG:", log_path, flush=True)
    t0, last = time.time(), time.time()
    with open(log_path, "w") as log:
        p = subprocess.Popen(list(map(str, cmd)), cwd=cwd, stdout=log, stderr=subprocess.STDOUT)
        while p.poll() is None:
            time.sleep(10)
            if time.time() - last >= heartbeat:
                print(f"  running {(time.time() - t0) / 60:.0f} min", flush=True); last = time.time()
    if p.returncode:
        raise RuntimeError(f"exit {p.returncode}; see {log_path}")
    print(f"done in {(time.time() - t0) / 60:.1f} min", flush=True)

def iter_fasta(path):
    name, chunks = None, []
    with (gzip.open(path, "rt") if str(path).endswith(".gz") else open(path)) as h:
        for line in h:
            line = line.rstrip("\n")
            if line.startswith(">"):
                if name is not None:
                    yield name, "".join(chunks)
                name, chunks = line[1:].split()[0], []
            else:
                chunks.append(line.strip())
    if name is not None:
        yield name, "".join(chunks)

def write_contigs(src_pairs, out_fa, prefix):
    """Нормализованный FASTA контигов для сравнения с эталоном."""
    n = 0
    tmp = Path(str(out_fa) + ".tmp")
    with open(tmp, "w") as h:
        for name, seq in src_pairs:
            seq = seq.upper()
            if seq:
                n += 1
                h.write(f">{prefix}_{n} {name}\n{seq}\n")
    tmp.replace(out_fa)
    print(out_fa, n, "contigs")
    return n

print("ROOT:", ROOT); print("BRANCH:", BRANCH_DIR); print("samples:", SAMPLES, "threads:", THREADS)


## Функции сравнения

In [ ]:
import csv, gzip, subprocess
from collections import defaultdict, Counter
from pathlib import Path


def open_text(path):
    path = str(path)
    return gzip.open(path, "rt") if path.endswith(".gz") else open(path)


def iter_fasta(path):
    name, chunks = None, []
    with open_text(path) as h:
        for line in h:
            line = line.rstrip("\n")
            if line.startswith(">"):
                if name is not None:
                    yield name, "".join(chunks)
                name, chunks = line[1:].split()[0], []
            else:
                chunks.append(line.strip())
    if name is not None:
        yield name, "".join(chunks)


def load_truth(branch_dir, sample):
    """Эталон V–J для каждого шаблона + класс восстановимости из QC симуляции."""
    meta = {r["template_id"]: r for r in csv.DictReader(
        open(branch_dir / "00_primary_truth" / f"{sample}_template_qc.tsv"), delimiter="\t")}
    cls = {r["template_id"]: r["reconstructability"] for r in csv.DictReader(
        open(branch_dir / "qc" / f"{sample}_template_reconstructability.tsv"), delimiter="\t")}
    truth = {}
    for tid, seq in iter_fasta(branch_dir / "00_primary_truth" / f"{sample}_templates.fasta"):
        m = meta[tid]
        copies = int(m.get("unique_umis") or m.get("observed_multiplicity") or 1)
        truth[tid] = {"locus": m["locus"], "vj": seq[int(m["vj_start"]):int(m["vj_end"])],
                      "class": cls.get(tid, "not_classified"),
                      "verified": m["verified"] == "True", "junction": m["junction"],
                      "copies": "1" if copies == 1 else ("2-4" if copies < 5 else "5+")}
    return truth



def minimap2_paf(reference, query, out_paf, threads):
    """Выравнивания с базовым разрешением (PAF с тегом NM) запросов на референс."""
    cmd = ["minimap2", "-c", "-x", "sr", "-N", "50", "-p", "0.5", "--secondary=yes",
           "-t", str(threads), str(reference), str(query)]
    with open(out_paf, "w") as out:
        subprocess.run(cmd, stdout=out, stderr=subprocess.DEVNULL, check=True)


def iter_paf(path):
    with open(path) as h:
        for line in h:
            f = line.rstrip("\n").split("\t")
            nm = next((int(x[5:]) for x in f[12:] if x.startswith("NM:i:")), 0)
            yield {"q": f[0], "qlen": int(f[1]), "qs": int(f[2]), "qe": int(f[3]),
                   "t": f[5], "tlen": int(f[6]), "ts": int(f[7]), "te": int(f[8]),
                   "match": int(f[9]), "block": int(f[10]), "nm": nm}


def recall_class(hit):
    """Класс эталонной V–J (запрос), выровненной на контиги (референс).

    exact_full — вся V–J без единой замены/инделя; one_diff_full — вся V–J, 1 отличие;
    near_full — >=95% V–J с идентичностью >=98% (мягкий порог, путает клональных родственников);
    partial — остальное с выравниванием; missing — выравнивания нет.
    """
    if hit is None:
        return "missing"
    cov = (hit["qe"] - hit["qs"]) / hit["qlen"]
    ident = hit["match"] / hit["block"]
    if cov == 1.0 and hit["nm"] == 0:
        return "exact_full"
    if cov == 1.0 and hit["nm"] == 1:
        return "one_diff_full"
    if cov >= 0.95 and ident >= 0.98:
        return "near_full"
    return "partial"


def precision_class(hit):
    """Класс контига (запрос), выровненного на эталонные V–J (референс)."""
    if hit is None:
        return "unmapped"
    ident = hit["match"] / hit["block"]
    tcov = (hit["te"] - hit["ts"]) / hit["tlen"]
    if tcov == 1.0 and hit["nm"] == 0:
        return "exact_full"
    if ident >= 0.98 and tcov >= 0.95:
        return "near_full"
    if ident >= 0.98 and hit["block"] >= 100:
        return "correct_partial"
    return "incorrect"


def best_by_query(paf_path, preferred=frozenset()):
    """Лучшее выравнивание каждого запроса; при равенстве предпочитается референс из `preferred`."""
    best, targets = {}, defaultdict(set)
    for h in iter_paf(paf_path):
        targets[h["q"]].add(h["t"])
        key = (h["match"] - h["nm"], h["qe"] - h["qs"], h["t"] in preferred)
        if h["q"] not in best or key > best[h["q"]][0]:
            best[h["q"]] = (key, h)
    return {q: v[1] for q, v in best.items()}, targets


def benchmark(truth, contigs, workdir, threads, native_cdr3=None):
    """Строки по шаблонам и по контигам. `contigs`: id -> последовательность."""
    workdir.mkdir(parents=True, exist_ok=True)
    truth_fa, contig_fa = workdir / "truth_vj.fasta", workdir / "contigs.fasta"
    with open(truth_fa, "w") as h:
        for tid, t in truth.items():
            h.write(f">{tid}\n{t['vj']}\n")
    with open(contig_fa, "w") as h:
        for cid, s in contigs.items():
            h.write(f">{cid}\n{s}\n")

    minimap2_paf(contig_fa, truth_fa, workdir / "truth_to_contigs.paf", threads)
    minimap2_paf(truth_fa, contig_fa, workdir / "contigs_to_truth.paf", threads)

    best_t, hit_contigs = best_by_query(workdir / "truth_to_contigs.paf")
    template_rows = []
    for tid, t in truth.items():
        h = best_t.get(tid)
        j = t.get("junction")
        cdr3 = "no_truth_junction" if not j else (
            "found" if any(j in contigs[c] for c in hit_contigs.get(tid, ())) else "not_found")
        row = {"template_id": tid, "locus": t["locus"], "reconstructability": t["class"], "copies": t["copies"],
               "verified": t["verified"],
               "junction": j or "",
               "recall": recall_class(h), "best_contig": h["t"] if h else "",
               "vj_coverage": round((h["qe"] - h["qs"]) / h["qlen"], 4) if h else 0,
               "identity": round(h["match"] / h["block"], 4) if h else 0,
               "edits": h["nm"] if h else "",
               "cdr3_in_contig": cdr3}
        if native_cdr3 is not None:
            row["cdr3_native"] = "no_truth_junction" if not j else ("found" if j in native_cdr3 else "not_found")
        template_rows.append(row)

    verified = frozenset(tid for tid, t in truth.items() if t["verified"])
    best_c, _ = best_by_query(workdir / "contigs_to_truth.paf", verified)
    contig_rows = []
    for cid, s in contigs.items():
        h = best_c.get(cid)
        cls = precision_class(h)
        if h and h["t"] not in verified and cls not in ("incorrect", "unmapped"):
            cls = "unverified_match"
        contig_rows.append({"contig_id": cid, "length": len(s), "precision": cls,
                            "best_template": h["t"] if h else "",
                            "identity": round(h["match"] / h["block"], 4) if h else 0})
    return template_rows, contig_rows


RECALL = ["exact_full", "one_diff_full", "near_full", "partial", "missing"]
PRECISION = ["exact_full", "near_full", "correct_partial", "unverified_match", "incorrect", "unmapped"]


def summarize(template_rows, contig_rows, group_keys=("reconstructability",)):
    """Recall по проверенным шаблонам в группах + строка UNVERIFIED (все непроверенные) + precision."""
    groups = defaultdict(list)
    for r in template_rows:
        if str(r["verified"]) != "True":
            groups[("UNVERIFIED",) * len(group_keys)].append(r)
            continue
        groups[tuple(r[k] for k in group_keys)].append(r)
        groups[("ALL",) * len(group_keys)].append(r)
    out = []
    for g, rows in sorted(groups.items()):
        n = len(rows)
        c = Counter(r["recall"] for r in rows)
        out.append({**dict(zip(group_keys, g)), "templates": n,
                    **{f"{k}_frac": round(c[k] / n, 4) for k in RECALL}})
    pc = Counter(r["precision"] for r in contig_rows)
    precision = {"contigs": len(contig_rows),
                 **{f"{k}_frac": round(pc[k] / max(1, len(contig_rows)), 4) for k in PRECISION}}
    return out, precision


def summarize_clonotypes(template_rows):
    """CDR3 на уровне клонотипов (локус + junction): клонотип «прочитан», если у любого его
    шаблона есть риды; отрицательный контроль — клонотипы, ни один шаблон которых не прочитан.
    Учитываются только проверенные шаблоны."""
    clon = defaultdict(list)
    for r in template_rows:
        if r["junction"] and str(r["verified"]) == "True":
            clon[(r["locus"], r["junction"])].append(r)
    groups = defaultdict(list)
    for key, rows in clon.items():
        state = "no_reads_control" if all(r["reconstructability"] == "no_reads" for r in rows) else "sequenced"
        rec = {"in_contig": any(r["cdr3_in_contig"] == "found" for r in rows),
               "native": any(r.get("cdr3_native") == "found" for r in rows)}
        groups[(key[0], state)].append(rec); groups[("ALL", state)].append(rec)
    out = []
    for (locus, state), recs in sorted(groups.items()):
        row = {"locus": locus, "clonotype_state": state, "clonotypes": len(recs),
               "cdr3_in_contig_frac": round(sum(x["in_contig"] for x in recs) / len(recs), 4)}
        if "cdr3_native" in template_rows[0]:
            row["cdr3_native_frac"] = round(sum(x["native"] for x in recs) / len(recs), 4)
        out.append(row)
    return out


def write_tsv(path, rows):
    if not rows:
        return
    with open(path, "w", newline="") as h:
        w = csv.DictWriter(h, fieldnames=list(rows[0]), delimiter="\t")
        w.writeheader()
        w.writerows(rows)


# самопроверка классификаторов
_h = {"qs": 0, "qe": 100, "qlen": 100, "match": 100, "block": 100, "nm": 0, "ts": 0, "te": 100, "tlen": 100}
assert recall_class(_h) == "exact_full" and precision_class(_h) == "exact_full"
assert recall_class({**_h, "match": 99, "nm": 1}) == "one_diff_full"
assert recall_class({**_h, "match": 98, "nm": 2}) == "near_full"
assert recall_class({**_h, "qe": 50}) == "partial" and recall_class(None) == "missing"
assert precision_class({**_h, "match": 90, "nm": 10}) == "incorrect" and precision_class(None) == "unmapped"


## Доступные результаты сборщиков

In [ ]:
TOOLS = ["trust4", "mixcr", "rnaspades", "trinity", "vdjer"]
BENCH_DIR = BRANCH_DIR / "benchmark"
available = {t: [s for s in SAMPLES if (ASM_DIR / t / s / "contigs.fasta").exists()] for t in TOOLS}
print({t: len(s) for t, s in available.items()})


## Сравнение с эталоном

In [ ]:
import csv
all_rows, precision_rows = {}, []
for tool, samples in available.items():
    for sample in samples:
        out = BENCH_DIR / tool / sample
        if (out / "per_template.tsv").exists() and not FORCE:
            rows = list(csv.DictReader(open(out / "per_template.tsv"), delimiter="\t"))
            crow = list(csv.DictReader(open(out / "per_contig.tsv"), delimiter="\t"))
        else:
            truth = load_truth(BRANCH_DIR, sample)
            contigs = dict(iter_fasta(ASM_DIR / tool / sample / "contigs.fasta"))
            rows, crow = benchmark(truth, contigs, out / "work", THREADS)
            write_tsv(out / "per_template.tsv", rows); write_tsv(out / "per_contig.tsv", crow)
        all_rows.setdefault(tool, []).extend(rows)
        _, prec = summarize(rows, crow)
        precision_rows.append({"tool": tool, "sample": sample, **prec})
        print(tool, sample, "done", flush=True)


## Доля точно восстановленных V…J (`exact_full`) по классам восстановимости

In [ ]:
import pandas as pd
summary = []
for tool, rows in all_rows.items():
    for keys in (("reconstructability",), ("copies",), ("locus",)):
        s, _ = summarize(rows, [], keys)
        for r in s:
            summary.append({"tool": tool, "group_by": keys[0], "group": r[keys[0]],
                            **{k: v for k, v in r.items() if k != keys[0]}})
    for r in summarize_clonotypes(rows):
        summary.append({"tool": tool, "group_by": "cdr3_clonotypes", "group": f"{r['locus']}:{r['clonotype_state']}",
                        "templates": r["clonotypes"], "cdr3_in_contig_frac": r["cdr3_in_contig_frac"]})
BENCH_DIR.mkdir(parents=True, exist_ok=True)
pd.DataFrame(summary).to_csv(BENCH_DIR / "assemblers_summary.tsv", sep="\t", index=False)
pd.DataFrame(precision_rows).to_csv(BENCH_DIR / "assemblers_precision.tsv", sep="\t", index=False)
df = pd.DataFrame(summary)
df[df.group_by == "reconstructability"].pivot_table(index="group", columns="tool", values="exact_full_frac")


## CDR3 по клонотипам (прочитанные и контроль)

In [ ]:
df[df.group_by == "cdr3_clonotypes"].pivot_table(index="group", columns="tool", values="cdr3_in_contig_frac")


## Precision контигов

In [ ]:
pd.DataFrame(precision_rows)
